In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict

In [3]:
load_dotenv()

True

In [4]:
model = ChatGoogleGenerativeAI(model="models/gemini-3.1-flash-lite-preview")

In [12]:
#define state
class CricketState(TypedDict):

    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundary_percent: float
    summary: str

In [17]:
#define logic for claculate strike rate
def calculate_sr(state: CricketState) -> CricketState:
    runs = state['runs']
    balls = state['balls']

    sr = (runs / balls) * 100

    state['sr'] = sr

    return {"sr" : sr }


In [18]:
#define logic for calculate bpb
def calculate_bpb(state: CricketState) -> CricketState:
    balls = state['balls']
    fours = state['fours']
    sixes = state['sixes']

    bpb = balls / (fours + sixes)

    state['bpb'] = bpb

    return {"bpb" : bpb}

In [19]:
#define logic for calculate boundary percent
def calculate_boundary_percent(state: CricketState) -> CricketState:
    runs = state['runs']
    fours = state['fours']
    sixes = state['sixes']

    boundary_percent = (((fours * 4) + (sixes * 6)) / runs) * 100

    state['boundary_percent'] = boundary_percent

    return {'boundary_percent' : boundary_percent}

In [20]:
#define logic to write summary
def calculate_summary(state: CricketState) -> CricketState:
    summary = f""""
    Strike Rate = {state['sr']} \n
    Balls per Boundary = {state['bpb']} \n
    Boundary Percentage = {state['boundary_percent']}
    """
    state['summary'] = summary

    return {"summary" : summary}


In [21]:
#initiate graph
graph = StateGraph(CricketState)

#define nodes, edges and compile
graph.add_node("calculate_sr", calculate_sr)
graph.add_node("calculate_bpb", calculate_bpb)
graph.add_node("calculate_boundary_percent", calculate_boundary_percent)
graph.add_node("calculate_summary", calculate_summary)

graph.add_edge(START, "calculate_sr")
graph.add_edge(START, "calculate_bpb")
graph.add_edge(START, "calculate_boundary_percent")

graph.add_edge("calculate_sr","calculate_summary")
graph.add_edge("calculate_bpb","calculate_summary")
graph.add_edge("calculate_boundary_percent","calculate_summary")

graph.add_edge("calculate_summary", END)

workflow = graph.compile()



In [23]:
initial_state = {
    "runs" : 100,
    "balls" : 75,
    "fours" : 6,
    "sixes" : 3
}

workflow.invoke(initial_state)
# print(final_state)

{'runs': 100,
 'balls': 75,
 'fours': 6,
 'sixes': 3,
 'sr': 133.33333333333331,
 'bpb': 8.333333333333334,
 'boundary_percent': 42.0,
 'summary': '"\n    Strike Rate = 133.33333333333331 \n\n    Balls per Boundary = 8.333333333333334 \n\n    Boundary Percentage = 42.0\n    '}